Ce *notebook* propose une rapide démonstration de certaines de fonctionnalités du paquet `lexploreur` à la date du 6 novembre 2026.

# Importation des textes

Le format d'entrée est un fichier CSV où chaque ligne est un document. Une colonne (nom libre) contient les textes des documents et il peut y avoir autant de colonnes que souhaitez pour décrire chaque document.

lexploreur attend un *dataframe* `pandas` contenant le CSV.

In [ ]:
import pandas as pd

In [ ]:
csvf = "bulletins_république.csv"

In [ ]:
df = pd.read_csv(csvf)

In [ ]:
df

# Installation modèle de langue spacy

On télécharge le modèle le plus léger pour le français.

In [ ]:
import spacy

In [ ]:
spacy.cli.download('fr_core_news_sm')

# Chargement du module corpus

C'est à partir de cette étape que l'on commence véritablement à utilsier le paquet lexploreur.

In [ ]:
from lexploreur.corpus import corpus

La fonction `corpus()` 

In [ ]:
corpus(df,
       corpus_name="bulletins_république.json",
       text_column="textes",
       nlp_model="fr_core_news_sm",
       ner=True,
       batch_size=16,
       n_process=4  # attention au nombre de coeurs dispo, -1 pour tous
      )

Une fois la fonction `corpus()` exécutée un fichier JSON contenant le corpus annoté et segmenté est enregistré. Ici c'est le fichier `bulletins_république.json`.

# Création d'une vue lexicale

Nous allons maintenant créer une vue lexicale qui construit une représentation du corpus avec les lemmes, les entités nommées et regroupe le documents en fonction des dates.

In [1]:
from lexploreur.corpus import lexical_view

lexploreur v. 0.2.0


In [2]:
jsonf = "bulletins_république.json"

In [3]:
vue = lexical_view(jsonf,
                   feature_to_extract="text",
                   group_by="date",
                   merge_ner=True)

Si  l'on affiche la variable `vue`, on constate qu'elle contient un *dataframe* `pandas` où les lignes correspondent à l'unité documentaire choisie (ici les dates) et où les unités lexicales de chaque unité documentaire sont stockées sous forme de liste.

In [ ]:
vue

# Représentation sac de mots

In [4]:
from lexploreur.representation import *

In [5]:
dtm = compute_dtm(vue, text_col="text", metadata_col="date")

Le *bag of words* est stockée sous la forme d'un *dataframe*.

In [ ]:
dtm

# Description des principales caractéristiques lexicométriques du corpus

In [ ]:
from lexploreur.description import *

On peut commencer par calculer le dictionnaire des fréquences :

In [ ]:
dico = features_frequencies(dtm)

In [ ]:
dico

On peut aussi observer la distribution des formes en fonction des fréquences et retrouver la fameuse loi de Zipf via une courbe de Pareto.

In [ ]:
plot_pareto(dico)

Il est facile d'obtenir la taille du corpus (ou plus précisément de la représentation construite de celui-ci dans la vue) :

In [ ]:
corpus_size(dtm)

Pour connaître la taille des différentes unités documentaires composant le corpus, on peut utiliser `group_sizes()` :

In [ ]:
pclc = group_sizes(dtm)

In [ ]:
pclc

`plot_group_sizes()` permet d'obtenir une représentation graphique de ce tableau.

In [ ]:
plot_group_sizes(pclc, "features")